## deep neural network

In [26]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

data_train = pd.read_csv('../data/train/train.csv')
data_test = pd.read_csv('../data/test/test.csv')

In [27]:
copy_data_train = data_train.copy()        
print(data_train.shape)
print(data_test.shape)

copy_data_train['CabinKnown'] = copy_data_train['Cabin'].notna()
copy_data_train['Age'] = copy_data_train['Age'].fillna(copy_data_train['Age'].median())
copy_data_train['Embarked'] = copy_data_train['Embarked'].fillna('S')

copy_data_train['CabinKnown'] = copy_data_train['CabinKnown'].astype('int32')
replace_dict = {'male': 0, 'female': 1}
copy_data_train['Sex'] = copy_data_train['Sex'].replace(replace_dict)
copy_data_train['Sex'] = copy_data_train['Sex'].astype('int32')

copy_data_train.head()


(891, 12)
(418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,0


In [29]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler


encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# data_procced_categories['Embarked'] = encoder.fit_transform(data_procced_categories['Embarked'])
encoded = encoder.fit_transform(copy_data_train[['Embarked']],)
encoded_df= pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Embarked"]),
    index=copy_data_train.index,
)

copy_data_train = pd.concat(
    [
        copy_data_train.drop(columns=['Embarked']),
        encoded_df
    ],
    axis=1
)

feature_to_scale = ['Age', 'Fare', 'SibSp', 'Parch']
scaler = StandardScaler()
copy_data_train[feature_to_scale] = scaler.fit_transform(copy_data_train[feature_to_scale])




In [34]:
import torch
from sklearn.model_selection import train_test_split
import torch.nn as nn

train_X = copy_data_train.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
train_y = copy_data_train['Survived']

x, x_test, y, y_test = train_test_split(train_X, train_y, test_size=0.2, random_state=42, shuffle=True)
X_train = torch.tensor(x.to_numpy(), dtype=torch.float64)
y_train = torch.tensor(y.to_numpy(), dtype=torch.float64)
x_test = torch.tensor(x_test.to_numpy(), dtype=torch.float64)
y_test = torch.from_numpy(y_test.to_numpy(dtype=np.float64))


In [46]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, X):
        return self.network(X)

obj = SimpleMLP(128, 64, 16)
xx = torch.rand(32, 128)
print(obj(xx).shape)

torch.Size([32, 16])
